# Anwendungsbeispiel: Generische Wahl

IPRES ist nicht auf die Bundestagswahl beschränkt. Dieses Notebook zeigt den vollständigen Workflow für eine beliebige Wahl — von der Konfiguration der Parteien und Wahlkreise bis zur Auswertung des Ergebnisses.

Für jede Konfiguration stehen zwei Wege zur Auswahl:
- **Option A (Interaktiv):** Widget-UI — nur in einem laufenden Jupyter-Server verfügbar.
- **Option B (Programmatisch):** Direkte Konfiguration im Code — läuft überall.

Für eine Erklärung des Wahlverfahrens: [Einführung](../../docs/source/de/einfuehrung.md)  
Für eine Simulation mit echten Wahldaten: [bundestagswahl.ipynb](bundestagswahl.ipynb)

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display

from ipres import (
    Election, ElectionConfig, Parties, ConstituenciesConfig,
    ElectionEvaluator,
)

---

## 1. Parteien konfigurieren

### Option A — Interaktiv

Die Widget-UI erlaubt es, Parteien aus einer Datei zu laden oder zufällig zu erzeugen.

In [ ]:
# Nur in laufendem Jupyter-Server: Widget-Eingabe erscheint unterhalb dieser Zelle
parties = Parties()
parties.setup()

### Option B — Programmatisch

In [ ]:
parties = Parties.from_dataframe(pd.DataFrame({
    'party_name': [
        'Fortschrittspartei',
        'Volksunion',
        'Grüne Zukunft',
        'Liberale Mitte',
        'Regionalpartei',
    ]
}))
display(parties.getParties())

---

## 2. Wahlkreise konfigurieren

### Option A — Interaktiv

In [ ]:
# Nur in laufendem Jupyter-Server
cc = ConstituenciesConfig()
cc.setup()

### Option B — Programmatisch

In [ ]:
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': [
        'Nordstadt', 'Südstadt', 'Ostkreis', 'Westkreis',
        'Mittelstadt', 'Bergdorf', 'Seestadt', 'Waldkreis',
    ],
    'constituency_size': [
        120_000, 95_000, 85_000, 110_000,
        75_000, 45_000, 130_000, 60_000,
    ],
    'turnout_percent': [
        72.0, 68.5, 74.2, 70.8,
        65.3, 71.1, 76.5, 69.4,
    ],
}))
display(cc.getConstituencies())

---

## 3. Wahlkonfiguration

Die `ElectionConfig` verbindet Parteien und Wahlkreise und berechnet automatisch Parlamentsgröße und Mehrheitsschwellen.

In [ ]:
party_names = parties.getPartyNames()
config = ElectionConfig(
    constituencies_config=cc,
    participating_parties=party_names,
    seed=42,
)
print(f"Parteien:        {party_names}")
print(f"Wahlkreise:      {cc.getM()}")
print(f"Parlamentssitze: {config.parliamentarySeats}")
print(f"Regierungssitze: {config.getParliamentMajoritySeats()}")
print(f"Ballotmehrheit:  {config.getBallotMajorityPercent():.1f} %")

---

## 4. Wahl durchführen

`election.run()` führt alle Runden automatisch durch, bis ein Gewinner feststeht. Der optionale Callback `on_iteration_finished` ermöglicht es, das Ergebnis jeder Runde zu verfolgen.

In [ ]:
rounds = []

def on_round_finished(iteration):
    rounds.append(iteration)
    n = len(rounds)
    if iteration.hasWinner():
        print(f"Runde {n}: Gewinner \u2192 {iteration.getWinner().name}")
    else:
        print(f"Runde {n}: kein Gewinner")
    display(iteration.show_results_table(styler=True))

election = Election(electionConfig=config)
final = election.run(on_iteration_finished=on_round_finished)

print(f"\nGesamtanzahl Runden: {election.getNumberOfIterations()}")
print(f"Wahlgewinner: {final.getWinner().name}")

---

## 5. Ergebnisauswertung

`ElectionEvaluator.evaluate()` führt die drei Auswertungsschritte automatisch durch:
Mandatszuteilung → Wahlkreisanzahlzuordnung → Wahlkreiszuordnung.

Details zu den einzelnen Schritten und ihren Konfigurationsoptionen: [`wahlauswertung.ipynb`](wahlauswertung.ipynb)

In [ ]:
evaluator = ElectionEvaluator(seed=42)
result = evaluator.evaluate(election)

In [ ]:
display(result.get_seat_distribution_table())

In [ ]:
display(result.get_constituency_summary_table())
display(result.get_constituency_assignment_table(sort_by='constituency'))

In [ ]:
fig = result.plot_seat_share_pie()
display(fig)
plt.close(fig)